# TFM 2 · Rebuild the reports from a state zip (CPU only)
Any state zip with `results/` (and/or `results_<output>/`) folders: the slim one, a new one from TFM 1, or an old Kaggle
zip like `analysis_B.zip`. It is unpacked into its own folder `results/tfm_bench_rebuild/<zip name>/`, so your live
results are never touched. Every metric is recomputed from the saved predictions; no model is rerun.

In [ ]:
# ---- settings ------------------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USER/ASML.git"   # <- your repo (Colab only; locally the notebook runs in place)
DRIVE_ROOT = "/content/drive/MyDrive/ASML"           # Colab only: Drive folder holding data/ and results/
STATE_ZIP = "../results/tfm_bench/benchmark_state_slim.zip"   # relative to notebooks/, an absolute path, or "upload"
NORM = "robust_z"          # default for the regressor-outputs page: robust_z | z | skill | rank | minmax
NORM_REF = "gpr_botorch"   # reference config for "skill"

In [ ]:
# Setup. Works in Colab (clones the repo, links data/ + results/ to Drive) and in local Jupyter (runs in place).
# Every path below is relative to the notebooks/ folder:  ../data   ../results/tfm_bench   tfm_bench/*.py
import os, sys, glob, shutil, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules


def secret(name):
    """Colab secret (key icon on the left) or, locally, an environment variable. None if missing."""
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/ASML"
    tok = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    if os.path.isdir(f"{REPO}/.git"):   # the Colab copy is disposable: make it match GitHub exactly
        subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "@{u}"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, REPO], check=True)
    with open(f"{REPO}/.git/info/exclude", "a") as fh:   # keep the Drive links out of git even if .gitignore says "data/"
        fh.write("\n/data\n/results\n")
    os.chdir(f"{REPO}/notebooks")
    for sub in ["data", "results"]:     # not in git: link them to Drive so ../data and ../results work
        os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
        if not os.path.islink(f"../{sub}"):
            if os.path.isdir(f"../{sub}"):
                shutil.rmtree(f"../{sub}")
            os.symlink(f"{DRIVE_ROOT}/{sub}", f"../{sub}")
else:
    assert os.path.isdir("tfm_bench"), "run this notebook from the repo's notebooks/ folder"

DATA, WORK = os.path.abspath("../data"), os.path.abspath("../results/tfm_bench")
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)


def restore_state(zip_path, work_dir, data_dir):
    """Unzip a state zip: data CSVs -> data_dir (only if missing), results*/ folders -> work_dir.
    Handles zips whose content sits under a folder (e.g. project/), like the old Kaggle ones."""
    tmp = os.path.abspath("../results/_state_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(tmp)
    hits = glob.glob(f"{tmp}/**/results*/*/results_*/predictions.csv", recursive=True)
    root = min((h.rsplit("/", 4)[0] for h in hits), key=len) if hits else tmp
    for sub in sorted(os.path.basename(p) for p in glob.glob(f"{root}/results*") if os.path.isdir(p)):
        shutil.copytree(f"{root}/{sub}", f"{work_dir}/{sub}", dirs_exist_ok=True)
    if os.path.exists(f"{root}/context_selection.csv"):
        shutil.copy(f"{root}/context_selection.csv", work_dir)
    for p in glob.glob(f"{tmp}/**/opt_results_*.csv", recursive=True):
        if not glob.glob(f"{data_dir}/**/{os.path.basename(p)}", recursive=True):
            shutil.copy(p, data_dir)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"restored {os.path.basename(zip_path)}: {len(glob.glob(f'{work_dir}/results*/*/results_*/predictions.csv'))} "
          f"results, {len(glob.glob(f'{data_dir}/**/opt_results_*.csv', recursive=True))} data CSVs")


print("notebooks dir:", os.getcwd(), "| data:", DATA, "| results:", WORK)

if STATE_ZIP == "upload":
    from google.colab import files
    up = files.upload(); zp = os.path.abspath(next(iter(up)))
else:
    zp = os.path.abspath(STATE_ZIP)
REBUILD = os.path.abspath(f"../results/tfm_bench_rebuild/{os.path.splitext(os.path.basename(zp))[0]}")
shutil.rmtree(REBUILD, ignore_errors=True); os.makedirs(REBUILD)
restore_state(zp, REBUILD, DATA)
os.environ["BENCH_WORK_DIR"] = REBUILD   # the report scripts read and write here
os.environ["BENCH_NORM_REF"] = NORM_REF

In [ ]:
!python tfm_bench/binomial_report.py                # die yield, every model + classifier fix, raw (no normalization)
!python tfm_bench/targets_report.py --norm {NORM}   # regressor outputs, normalization switchable in the page
!python tfm_bench/analyze.py                        # original yield summary (curves over the run, flagged errors)
print("pages in", f"{REBUILD}/avg_everything")

In [ ]:
# Colab: download the pages (self-contained) + tables. Locally they're already in the folder printed above.
if IN_COLAB:
    from google.colab import files
    for f in ["binomial_comparison.html", "binomial_metrics.csv", "targets_comparison.html", "targets_rankings_all_norms.csv",
              "summary.html"]:
        if os.path.exists(f"{REBUILD}/avg_everything/{f}"):
            files.download(f"{REBUILD}/avg_everything/{f}")